In [ ]:
# from google.colab import files

# # Upload preprocessing.py
# uploaded = files.upload()

In [1]:
import sys
sys.path.append('./code')

In [2]:
import torch
from transformers import BertTokenizerFast
from encoder import Encoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load tokenizer
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# Load Encoder model
encoder = Encoder(
    vocab_size=tokenizer.vocab_size,
    hidden_size=256,
    num_heads=4,
    num_layers=4,
    intermediate_size=512,
    max_len=64
).to(device)

encoder.load_state_dict(torch.load('code/mlm/best_encoder_checkpoint.pt', map_location=device))
encoder.eval()
print("Loaded trained Encoder checkpoint successfully.")

# Load raw_text.pkl
import pickle

with open('data/raw_text.pkl', 'rb') as f:
    raw_text = pickle.load(f)

print(f"Loaded {len(raw_text)} stories.")


Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded trained Encoder checkpoint successfully.
Loaded 109 stories.


<ipython-input-2-15b16746ec62>:29: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  raw_text = pickle.load(f)


In [3]:
import os
import pickle
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Huggingface
from transformers import BertTokenizerFast

# Your own modules
from encoder import Encoder
from data import TextDataset
from preprocessing import downsample_word_vectors, make_delayed

In [4]:
def generate_word_level_encoder_features(raw_text, tokenizer, encoder, device, max_len=16, batch_size=256):
    """
    Generates word-level encoder embeddings for each story, using small batches to avoid CUDA OOM.

    Args:
        raw_text: dict of story_id -> DataSequence
        tokenizer: Huggingface tokenizer
        encoder: trained encoder model
        device: cpu or cuda
        max_len: Maximum token length per word
        batch_size: Batch size for feeding into encoder to save memory

    Returns:
        word_vectors: dict of story_id -> [num_words, hidden_size]
    """
    encoder.eval()
    word_vectors = {}

    for story_id, seq in tqdm(raw_text.items(), desc="Encoding stories"):
        words = seq.data  # list of words
        embeddings_list = []  # collect all batch outputs here

        for start_idx in range(0, len(words), batch_size):
            end_idx = min(start_idx + batch_size, len(words))
            batch_words = words[start_idx:end_idx]

            # Tokenize a small batch
            encodings = tokenizer(
                batch_words,
                padding="max_length",
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            )

            input_ids = encodings["input_ids"].to(device)
            token_type_ids = encodings["token_type_ids"].to(device)
            attention_mask = encodings["attention_mask"].unsqueeze(1).unsqueeze(1).to(device)

            with torch.no_grad():
                outputs = encoder(input_ids, token_type_ids, attention_mask, output_logits=False)  # (batch_size, seq_len, hidden_size)

            # Take the CLS token's output
            embeddings = outputs[:, 0, :]  # (batch_size, hidden_size)

            embeddings_list.append(embeddings.cpu().numpy())

        # After all batches are processed, concatenate embeddings
        story_embedding = np.concatenate(embeddings_list, axis=0)  # (num_words, hidden_size)

        word_vectors[story_id] = story_embedding

    return word_vectors


def downsample_features(word_level_features, raw_text):
    story_ids = list(word_level_features.keys())
    downsampled = downsample_word_vectors(
        stories=story_ids,
        word_vectors=word_level_features,
        wordseqs=raw_text
    )
    return downsampled

def trim_features(downsampled_features, TR=1, skip_seconds=(5, 10)):
    trim_start = int(skip_seconds[0] / TR)
    trim_end = int(skip_seconds[1] / TR)

    trimmed = {}
    for story_id, X in downsampled_features.items():
        if X.shape[0] > (trim_start + trim_end):
            X_trimmed = X[trim_start:-trim_end]
            trimmed[story_id] = X_trimmed
        else:
            print(f"Skipped {story_id}: too short after trimming ({X.shape[0]} rows)")

    return trimmed

def apply_delays(trimmed_features, delays=range(1, 5)):
    delayed = {}
    for story_id, X in trimmed_features.items():
        X_delayed = make_delayed(X, delays=delays)
        delayed[story_id] = X_delayed
    return delayed

In [5]:
# Generate encoder-based word vectors
story_to_X_words = generate_word_level_encoder_features(raw_text, tokenizer, encoder, device)

Encoding stories: 100%|██████████| 109/109 [00:07<00:00, 13.81it/s]


In [6]:
for story_id, X in story_to_X_words.items():
    print(f"{story_id}: word-level shape = {X.shape}")

sweetaspie: word-level shape = (697, 256)
thatthingonmyarm: word-level shape = (2073, 256)
tildeath: word-level shape = (2297, 256)
indianapolis: word-level shape = (1554, 256)
lawsthatchokecreativity: word-level shape = (2084, 256)
golfclubbing: word-level shape = (1211, 256)
jugglingandjesus: word-level shape = (887, 256)
shoppinginchina: word-level shape = (1731, 256)
cocoonoflove: word-level shape = (1984, 256)
hangtime: word-level shape = (1927, 256)
beneaththemushroomcloud: word-level shape = (1916, 256)
dialogue4: word-level shape = (1692, 256)
thepostmanalwayscalls: word-level shape = (2220, 256)
stumblinginthedark: word-level shape = (2681, 256)
kiksuya: word-level shape = (1699, 256)
haveyoumethimyet: word-level shape = (2985, 256)
theinterview: word-level shape = (1079, 256)
againstthewind: word-level shape = (838, 256)
tetris: word-level shape = (1350, 256)
canplanetearthfeedtenbillionpeoplepart2: word-level shape = (2532, 256)
alternateithicatom: word-level shape = (2174, 

In [7]:
story_to_X_ds = downsample_features(story_to_X_words, raw_text)

In [8]:
for story_id, X in story_to_X_ds.items():
    print(f"{story_id}: word-level shape = {X.shape}")

sweetaspie: word-level shape = (172, 256)
thatthingonmyarm: word-level shape = (449, 256)
tildeath: word-level shape = (338, 256)
indianapolis: word-level shape = (317, 256)
lawsthatchokecreativity: word-level shape = (449, 256)
golfclubbing: word-level shape = (216, 256)
jugglingandjesus: word-level shape = (208, 256)
shoppinginchina: word-level shape = (352, 256)
cocoonoflove: word-level shape = (444, 256)
hangtime: word-level shape = (339, 256)
beneaththemushroomcloud: word-level shape = (357, 256)
dialogue4: word-level shape = (322, 256)
thepostmanalwayscalls: word-level shape = (469, 256)
stumblinginthedark: word-level shape = (504, 256)
kiksuya: word-level shape = (347, 256)
haveyoumethimyet: word-level shape = (511, 256)
theinterview: word-level shape = (236, 256)
againstthewind: word-level shape = (185, 256)
tetris: word-level shape = (295, 256)
canplanetearthfeedtenbillionpeoplepart2: word-level shape = (560, 256)
alternateithicatom: word-level shape = (358, 256)
goldiethegold

In [9]:
story_to_X_trimmed = trim_features(story_to_X_ds)

In [10]:
for story_id, X in story_to_X_trimmed.items():
    print(f"{story_id}: word-level shape = {X.shape}")

sweetaspie: word-level shape = (157, 256)
thatthingonmyarm: word-level shape = (434, 256)
tildeath: word-level shape = (323, 256)
indianapolis: word-level shape = (302, 256)
lawsthatchokecreativity: word-level shape = (434, 256)
golfclubbing: word-level shape = (201, 256)
jugglingandjesus: word-level shape = (193, 256)
shoppinginchina: word-level shape = (337, 256)
cocoonoflove: word-level shape = (429, 256)
hangtime: word-level shape = (324, 256)
beneaththemushroomcloud: word-level shape = (342, 256)
dialogue4: word-level shape = (307, 256)
thepostmanalwayscalls: word-level shape = (454, 256)
stumblinginthedark: word-level shape = (489, 256)
kiksuya: word-level shape = (332, 256)
haveyoumethimyet: word-level shape = (496, 256)
theinterview: word-level shape = (221, 256)
againstthewind: word-level shape = (170, 256)
tetris: word-level shape = (280, 256)
canplanetearthfeedtenbillionpeoplepart2: word-level shape = (545, 256)
alternateithicatom: word-level shape = (343, 256)
goldiethegold

In [11]:
story_to_X_lagged = apply_delays(story_to_X_trimmed, delays=range(1, 5))

In [12]:
for story_id, X in story_to_X_lagged.items():
    print(f"{story_id}: word-level shape = {X.shape}")

sweetaspie: word-level shape = (157, 1024)
thatthingonmyarm: word-level shape = (434, 1024)
tildeath: word-level shape = (323, 1024)
indianapolis: word-level shape = (302, 1024)
lawsthatchokecreativity: word-level shape = (434, 1024)
golfclubbing: word-level shape = (201, 1024)
jugglingandjesus: word-level shape = (193, 1024)
shoppinginchina: word-level shape = (337, 1024)
cocoonoflove: word-level shape = (429, 1024)
hangtime: word-level shape = (324, 1024)
beneaththemushroomcloud: word-level shape = (342, 1024)
dialogue4: word-level shape = (307, 1024)
thepostmanalwayscalls: word-level shape = (454, 1024)
stumblinginthedark: word-level shape = (489, 1024)
kiksuya: word-level shape = (332, 1024)
haveyoumethimyet: word-level shape = (496, 1024)
theinterview: word-level shape = (221, 1024)
againstthewind: word-level shape = (170, 1024)
tetris: word-level shape = (280, 1024)
canplanetearthfeedtenbillionpeoplepart2: word-level shape = (545, 1024)
alternateithicatom: word-level shape = (343

In [14]:
import os
import time
import joblib

# 1. Set target save path
save_dir = '../data'
save_filename = 'X_lagged_mlm.joblib'
save_path = os.path.join(save_dir, save_filename)

# 2. Make sure the directory exists
os.makedirs(save_dir, exist_ok=True)

# 3. Start saving
print(f"Saving story_to_X_lagged to {save_path} using joblib...")

start_time = time.time()

# Dump the object
joblib.dump(story_to_X_lagged, save_path)

elapsed = time.time() - start_time

# 4. After saving, print file size
file_size = os.path.getsize(save_path) / (1024 * 1024)  # in MB
print(f"Saved successfully! File size: {file_size:.2f} MB. Time taken: {elapsed:.2f} seconds.")

Saving story_to_X_lagged to ../data/X_lagged_mlm.joblib using joblib...
Saved successfully! File size: 291.79 MB. Time taken: 0.26 seconds.
